## Colab session configuration

Connect to google cloud runtime and use GPU.

## Reading hg38 genome assembly 

Import.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, pathlib, subprocess
from pathlib import Path
!pip install pyfaidx
from pyfaidx import Fasta
from huggingface_hub import hf_hub_download

c:\Users\eddyk\miniconda3\envs\staix\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configure root.

In [2]:
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")

if COLAB:
    root = Path("/content/mini-gLM")

    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])

else:
    root = Path.cwd().parent

sys.path.insert(0, str(root))

Load pre-training data from HF.

In [ ]:
path = hf_hub_download(
    repo_id="eddykang06/hg38-pretraining",
    repo_type="dataset",
    filename="pretraining.csv.gz",
)

pretraining = pd.read_csv(path, compression = "gzip", usecols = ["sequence"])["sequence"].to_list()

c:\Users\eddyk\miniconda3\envs\staix\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\eddyk\.cache\huggingface\hub\datasets--eddykang06--hg38-pretraining. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


## Tokenizer

Train BPE tokenizer.

In [ ]:
from src.tokenize import BPETokenizer

tokenizer = BPETokenizer()

## Model architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Implement relative positional encoding.

In [1]:
class ALiBi(nn.Module):
    def __init__(self):
        super().__init__()
    

NameError: name 'nn' is not defined

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = self.d_model // self.num_heads

        # Queries, keys, values
        self.q_map = nn.Linear(d_model, d_model)
        self.k_map = nn.Linear(d_model, d_model)
        self.v_map = nn.Linear(d_model, d_model)

        # Final FC
        self.o_map = nn.Linear(d_model, d_model)

    def forward(self, x, mask):

        B, L, D = x.shape

        q = self.q_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        k = self.k_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)
        v = self.v_map(x).reshape(B, L, self.num_heads, self.d_head).transpose(1, 2)

        scores = q @ k.transpose(-2, -1) / (self.d_head ** 0.5)

        # Padding mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        a = torch.softmax(scores, dim = -1)
        
        out = a @ v
        out = out.transpose(1, 2).reshape(B, L, D)
        out = self.o_map(out)

        return out

# Note: make sure tha mask is [B, 1, 1, L]

Implement multi-head sparse attention block with attention mask.

In [ ]:
class MultiHeadSparseAttention():
    def __init__(self, d_model, num_heads):
        super().__init__()

Implement Mixture of Experts block.

- https://huggingface.co/blog/moe
- https://machinelearningmastery.com/mixture-of-experts-architecture-in-transformer-models/

In [ ]:
# SwiGLU style expert
class SwiGLU(nn.Module):
    def __init__(self, input_dim, h_dim):
        super().__init__()

        self.input_dim = input_dim
        self.h_dim = h_dim

        self.gate_proj = nn.Linear(input_dim, h_dim)
        self.up_proj = nn.Linear(input_dim, h_dim)
        self.down_proj = nn.Linear(h_dim, input_dim)
        self.act = nn.SiLU()

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        swish = self.act(gate)
        out = self.down_proj(swish * up)

        return out
    

class MoELayer(nn.Module):
    def __init__(self, input_dim, h_dim, num_experts, top_k):
        super().__init__()
        
        self.input_dim = input_dim
        self.h_dim = h_dim,
        self.num_experts = num_experts
        self.top_k = top_k

        # Initialize the swiglu experts
        self.experts = nn.ModuleList([
            SwiGLU(input_dim, h_dim) for _ in range(num_experts)
        ])

        # Router for per-expert logits
        self.router = nn.Linear(input_dim, num_experts)

    def forward(self, x):

        B, L, D = x.shape

        # Reshape for expert processing [B * L, D]
        x_reshaped = x.reshape(-1, D)

        # Logits to [B * L, num_experts]
        router_logits = self.router(x_reshaped)

        # Get the top-k experts, then softmax to probabilty distribution over those k experts
        # Output [B * L, k]
        top_k_logits, top_k_idx = torch.topk(router_logits, self.top_k, dim = -1)
        top_k_probs = F.softmax(top_k_logits, dim = -1)

        # Initialize output tensor
        out = torch.zeros(
            B * L, D,
            device = x.device,
            dtype = x.dtype
        )

        # Process through selected experts
        unique_experts = torch.unique(top_k_idx)

        for i in unique_experts:
            expert_id = int(i)

            # Token mask [B*L] to decide which token of input should use this expert
            mask = (top_k_idx == expert_id)
            token_mask = mask.any(dim = 1)
            assert token_mask.any()

            # Select tokens, apply the expert, and add to output
            expert_input = x_reshaped[token_mask]
            expert_weight = top_k_probs[mask].unsqueeze(-1) # [N, 1]
            expert_output = self.experts[expert_id](expert_input) # [N, hidden_dim]

            out[token_mask] += expert_output * expert_weight

        # Reshape to original
        out = out.reshape(B, L, D)

        return out

Implement transformer encoder block with mixture of experts.

In [ ]:
class MoETransformer():
    def __init__(self, d_model, num_heads, h_dim, num_experts, top_k, p_drop):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.hidden_dim = h_dim
        self.num_experts = num_experts
        self.top_k = top_k

        # Layers
        self.attention = MultiHeadAttention()
        self.dropout1 = nn.Dropout(p = p_drop)
        self.norm1 = nn.LayerNorm()
        self.moe = MoELayer(
            input_dim = self.d_model,
            h_dim = self.h_dim,
            num_experts = self.num_experts,
            top_k = self.top_k
        )
        self.dropout2 = nn.Dropout(p = p_drop)
        self.norm2 = nn.LayerNorm()
    
    def forward(self, x):

        x = self.norm1(x + self.dropout1(self.attention(x))) 
        out = self.norm2(x + self.dropout2(self.moe(x)))

        return out

Implement transformer block with sparse attention.

In [ ]:
class SparseMoETransformer():
    def __init__(self):
        super().__init__()

Implement models.

In [ ]:
class GLM(nn.Module):




    # relative position encoding using alibi

## Training

Implement masked language modeling training objective.
- Embedding module
- Relative positional embeddings
- Batching [B, L, D]
    - Token right padding to max length in batch
- Attention mask to ignore padding tokens
- Max sequence length cutoff
- Masked token prediction objective
Reference: https://huggingface.co/learn/llm-course/en/chapter6/5